# 3D Poisson via Kronecker Factorisation

This notebook solves the 3D Poisson problem
$$-\Delta u = 1 \quad \text{in } \Omega = [0,1]^3, \qquad u = 0 \quad \text{on } \partial\Omega$$
using two alternative Kronecker decompositions of the same global operator.

## Decompositions compared

| Case | Subdomains | Global matrix |
|------|-----------|---------------|
| **2D × 1D** | `Ω_xy` (2D) + `Ω_z` (1D) | `K_xy ⊗ M_z + M_xy ⊗ K_z` |
| **1D × 1D × 1D** | `Ω_x`, `Ω_y`, `Ω_z` (all 1D) | `K_x⊗M_y⊗M_z + M_x⊗K_y⊗M_z + M_x⊗M_y⊗K_z` |

Both decompositions yield the **same global operator** on a uniform Cartesian grid.
We verify this by comparing the assembled matrices and their spectra.

The 2D×1D case is assembled manually using standard Gridap for the 2D part.
The 1D×1D×1D case uses the full `TensorProductAffineOperator` pipeline.

In [1]:
using Gridap
using GridapTensorProduct
using LinearAlgebra
using SparseArrays

---
## Shared parameters

In [2]:
# Grid resolution and discretisation
N = 6          # cells per direction (N+1 nodes, N-1 interior nodes for P1)
quad_order = 2
reffe = ReferenceFE(lagrangian, Float64, 1)

nf = N - 1     # free DOFs per 1D subdomain
n_1d3 = nf^3   # total free DOFs for 1D×1D×1D decomposition
n_2d1 = nf^2 * nf  # same (2D part has nf² free DOFs × nf for z)

println("Grid: ", N, "×", N, "×", N, " → ", nf, "³ = ", n_1d3, " free DOFs")

Grid: 6×6×6 → 5³ = 125 free DOFs


---
## Case 1: 2D × 1D decomposition

We split the 3D domain as $\Omega = \Omega_{xy} \times \Omega_z$ where:
- $\Omega_{xy} = [0,1]^2$ — a 2D Cartesian mesh assembled with standard Gridap
- $\Omega_z = [0,1]$ — a 1D subdomain

The Kronecker decomposition of the Laplacian is:
$$A = K_{xy} \otimes M_z + M_{xy} \otimes K_z$$
where $K_{xy}$ and $M_{xy}$ are the standard 2D stiffness and mass matrices,
and $K_z$, $M_z$ are the 1D matrices on $\Omega_z$.

In [3]:
# ── 2D subdomain (xy-plane) ──────────────────────────────────────────────
model_xy = CartesianDiscreteModel((0.0, 1.0, 0.0, 1.0), (N, N))
Ωxy = Interior(model_xy)
dΩxy = Measure(Ωxy, quad_order)

V_xy = TestFESpace(Ωxy, reffe; conformity=:H1, dirichlet_tags="boundary")
U_xy = TrialFESpace(V_xy, 0.0)

nf_xy = num_free_dofs(V_xy)   # = (N-1)² = 25
println("2D subdomain: ", num_cells(Ωxy), " cells, ", nf_xy, " free DOFs")

# Extract 2D stiffness and mass matrices
K_xy = Matrix(get_matrix(AffineFEOperator(
    (u, v) -> ∫(∇(u) ⋅ ∇(v)) * dΩxy,
    (v)    -> ∫(0*v) * dΩxy,
    U_xy, V_xy)))

M_xy = Matrix(get_matrix(AffineFEOperator(
    (u, v) -> ∫(u * v) * dΩxy,
    (v)    -> ∫(0*v) * dΩxy,
    U_xy, V_xy)))

println("K_xy size: ", size(K_xy), "  M_xy size: ", size(M_xy))

2D subdomain: 36 cells, 25 free DOFs
K_xy size: (25, 25)  M_xy size: (25, 25)


In [4]:
# ── 1D z subdomain ───────────────────────────────────────────────────────
model_z = CartesianDiscreteModel((0.0, 1.0), N)
Ωz = Interior(model_z)
dΩz = Measure(Ωz, quad_order)

V_z = TestFESpace(Ωz, reffe; conformity=:H1, dirichlet_tags="boundary")
U_z = TrialFESpace(V_z, 0.0)

nf_z = num_free_dofs(V_z)   # = N-1 = 5
println("1D subdomain z: ", num_cells(Ωz), " cells, ", nf_z, " free DOFs")

K_z = Matrix(get_matrix(AffineFEOperator(
    (u, v) -> ∫(∇(u) ⋅ ∇(v)) * dΩz,
    (v)    -> ∫(0*v) * dΩz,
    U_z, V_z)))

M_z = Matrix(get_matrix(AffineFEOperator(
    (u, v) -> ∫(u * v) * dΩz,
    (v)    -> ∫(0*v) * dΩz,
    U_z, V_z)))

println("K_z size: ", size(K_z), "  M_z size: ", size(M_z))

1D subdomain z: 6 cells, 5 free DOFs
K_z size: (5, 5)  M_z size: (5, 5)


In [5]:
# ── Kronecker assembly: A = K_xy ⊗ M_z + M_xy ⊗ K_z ────────────────────
# Convention: Ω_xy = subdomain 1 (fastest-varying DOF), Ω_z = subdomain 2
A_2d1d = kron(M_z, K_xy) + kron(K_z, M_xy)

# RHS: b = kron(b_z, b_xy)
b_xy = Vector(get_vector(AffineFEOperator(
    (u, v) -> ∫(0*u*v) * dΩxy, (v) -> ∫(1*v) * dΩxy, U_xy, V_xy)))
b_z  = Vector(get_vector(AffineFEOperator(
    (u, v) -> ∫(0*u*v) * dΩz,  (v) -> ∫(1*v) * dΩz,  U_z,  V_z)))
b_2d1d = kron(b_z, b_xy)

u_2d1d = A_2d1d \ b_2d1d

println("A_2d1d size: ", size(A_2d1d))
println("Max solution: ", round(maximum(u_2d1d), digits=6))

A_2d1d size: (125, 125)
Max solution: 0.058759


---
## Case 2: 1D × 1D × 1D decomposition

The same problem is solved with three independent 1D subdomains:
$$A = K_x \otimes M_y \otimes M_z  +  M_x \otimes K_y \otimes M_z  +  M_x \otimes M_y \otimes K_z$$

Using the `TensorProductAffineOperator` high-level API.

In [6]:
# Three identical 1D subdomains on [0,1]
model_x = CartesianDiscreteModel((0.0, 1.0), N)
model_y = CartesianDiscreteModel((0.0, 1.0), N)
# model_z defined above — reuse

Ωx = Interior(model_x)
Ωy = Interior(model_y)
dΩx = Measure(Ωx, quad_order)
dΩy = Measure(Ωy, quad_order)

Vx = TestFESpace(Ωx, reffe; conformity=:H1, dirichlet_tags="boundary")
Vy = TestFESpace(Ωy, reffe; conformity=:H1, dirichlet_tags="boundary")
Ux = TrialFESpace(Vx, 0.0)
Uy = TrialFESpace(Vy, 0.0)

println("1D subdomains: nf_x=", num_free_dofs(Vx),
        ", nf_y=", num_free_dofs(Vy),
        ", nf_z=", num_free_dofs(V_z))

1D subdomains: nf_x=5, nf_y=5, nf_z=5


In [7]:
# Build tensor product spaces
V3 = TensorProductFESpace(Vx, Vy, V_z)
U3 = TensorProductFESpace(Ux, Uy, U_z)

println("Total free DOFs: ", num_free_dofs(V3))   # nf^3

Total free DOFs: 125


In [8]:
# High-level operator
a(u, v) = nothing   # symbolic intent; assembly driven by op_type
l(v)    = nothing

op3 = TensorProductAffineOperator(a, l, U3, V3;
                                   op_type    = :stiffness,
                                   quad_order = quad_order)

A_1d3 = get_matrix(op3)
b_1d3 = get_vector(op3)
u_1d3 = A_1d3 \ b_1d3

println("A_1d3 size:   ", size(A_1d3))
println("Max solution: ", round(maximum(u_1d3), digits=6))

A_1d3 size:   (125, 125)
Max solution: 0.058759


---
## Comparison: are the two decompositions equivalent?

On a uniform Cartesian grid the two decompositions represent the **same** bilinear form.
The DOF orderings may differ, but the spectra and solution norms must match exactly.

In [9]:
# ── Sorted eigenvalue comparison ─────────────────────────────────────────
λ_2d1d = sort(eigvals(A_2d1d))
λ_1d3  = sort(eigvals(A_1d3))

println("Number of eigenvalues: ", length(λ_2d1d))
println("Max |λ_2d1d - λ_1d3|: ", maximum(abs, λ_2d1d - λ_1d3))

# The sorted spectra should agree to machine precision
@assert maximum(abs, λ_2d1d - λ_1d3) < 1e-10  "Spectra differ — decompositions inconsistent!"
println("PASS: spectra match to machine precision")

Number of eigenvalues: 125
Max |λ_2d1d - λ_1d3|: 8.881784197001252e-16
PASS: spectra match to machine precision


In [10]:
# ── Solution norm comparison ─────────────────────────────────────────────
# Both solve the same problem; norms should match even though DOF orderings differ
println("||u_2d1d||₂ = ", round(norm(u_2d1d), digits=8))
println("||u_1d3 ||₂ = ", round(norm(u_1d3),  digits=8))
println("Relative norm difference: ",
    abs(norm(u_2d1d) - norm(u_1d3)) / norm(u_1d3))

||u_2d1d||₂ = 0.38377978
||u_1d3 ||₂ = 0.38377978
Relative norm difference: 0.0


In [11]:
# ── Matrix structure comparison ───────────────────────────────────────────
# Both matrices have the same sparsity pattern (up to DOF permutation on the 2D side).
# We check that the Frobenius norms agree.
println("||A_2d1d||_F = ", round(norm(A_2d1d), digits=6))
println("||A_1d3 ||_F = ", round(norm(A_1d3),  digits=6))
println("Both symmetric: ",
    maximum(abs, A_2d1d - A_2d1d') < 1e-12,
    " / ",
    maximum(abs, A_1d3 - A_1d3') < 1e-12)

||A_2d1d||_F = 5.052808
||A_1d3 ||_F = 5.052808
Both symmetric: true / true


---
## Memory comparison: lazy vs eager

The key advantage of the Kronecker approach is that the global matrix never needs
to be materialised in memory — the structure can be exploited directly.

For a problem with $n$ DOFs per direction and $N$ directions:
- **Dense global matrix:** $n^N \times n^N$ entries ($O(n^{2N})$ memory)
- **Kronecker factors:** $N$ matrices of size $n \times n$ ($O(N n^2)$ memory)

In [12]:
# Memory footprint comparison for the 3D problem
n_dof = num_free_dofs(V_z)   # DOFs per direction
n_tot = n_dof^3              # total DOFs

mem_global_bytes = n_tot^2 * 8  # Float64 dense matrix
mem_factors_bytes = 3 * n_dof^2 * 8  # three n×n factor matrices

println("Total DOFs:         ", n_tot)
println("Dense global matrix: ", round(mem_global_bytes / 1024, digits=1), " KB")
println("Kronecker factors:   ", round(mem_factors_bytes / 1024, digits=1), " KB")
println("Memory ratio:        ", round(mem_global_bytes / mem_factors_bytes, digits=0), ":1")

println("\nFor n=100 per direction (100³ DOFs):")
n_large = 100; n_large_tot = n_large^3
dense_GB  = n_large_tot^2 * 8 / 1e9
kronecker_MB = 3 * n_large^2 * 8 / 1e6
println("  Dense:     ", round(dense_GB, digits=0), " GB")
println("  Kronecker: ", round(kronecker_MB, digits=2), " MB")

Total DOFs:         125
Dense global matrix: 122.1 KB
Kronecker factors:   0.6 KB
Memory ratio:        208.0:1

For n=100 per direction (100³ DOFs):
  Dense:     8000.0 GB
  Kronecker: 0.24 MB


---
## Summary

| | 2D × 1D | 1D × 1D × 1D |
|---|---|---|
| Subdomains | 1 × 2D + 1 × 1D | 3 × 1D |
| Assembly | Manual `kron` from Gridap matrices | `TensorProductAffineOperator` |
| Kronecker terms | 2 (`K_xy⊗M_z`, `M_xy⊗K_z`) | 3 (`K_x⊗M_y⊗M_z`, ...) |
| Spectra | identical | identical |
| Advantage | Fewer Kronecker terms (useful when xy has fine-scale structure) | Full separability; library handles everything |

The 1D×1D×1D decomposition is fully automated by GridapTensorProduct and requires
only adding a third subdomain to the `TensorProductFESpace`. The 2D×1D decomposition
shows how to mix standard Gridap assembly with manual Kronecker products for cases
where one subdomain is intrinsically 2D (e.g. a full 2D mesh with non-separable geometry).